# Encoding Categorical Data — Ordinal & Label Encoding

Most ML models need numbers, not text — **encoding** converts categorical columns into numeric ones. This notebook covers two encoders for two different situations:
- **`OrdinalEncoder`** — for an input feature with a natural **order** (e.g. `Poor` < `Average` < `Good`)
- **`LabelEncoder`** — for encoding the **target column** specifically (works for any number of categories, order or no order)

## 1. Load and Prepare the Data

`customer.csv` has columns `age, gender, review, education, purchased`. Dropping the first two columns (`age`, `gender`) leaves `review` and `education` as features, and `purchased` as the target.

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('customer.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'customer.csv'

In [ ]:
df.sample(5)

In [ ]:
df = df.iloc[:,2:]

### ⚠️ Important: `X_train` / `y_train` Were Never Created

The original notebook jumps straight to using `X_train` (in the `OrdinalEncoder` step) and `y_train` (in the `LabelEncoder` step) — but neither was ever defined anywhere. Running the original cells top to bottom raises `NameError: name 'X_train' is not defined` immediately. The missing step is `train_test_split()`, splitting the features (`review`, `education`) from the target (`purchased`) and setting aside a test set — added here.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('purchased', axis=1),
    df['purchased'],
    test_size=0.2,
    random_state=42
)


In [ ]:
df.head()

`OrdinalEncoder` maps each category to an integer based on **rank**. Without `categories=`, it would assign numbers alphabetically (`Average`→0, `Good`→1, `Poor`→2) — which is wrong here, since it doesn't reflect the real order (`Poor` < `Average` < `Good`). Passing `categories=` explicitly, in the correct order for each column, fixes that.

In [ ]:
X_train


In [ ]:
X_train

### Shortcut: `fit_transform()`

Same convenience method covered in the scaling notebooks — combines `.fit()` and `.transform()` on the same data into one call.

In [ ]:
# X_train_encoded = oe.fit_transform(X_train)
# equivalent to: oe.fit(X_train); X_train_encoded = oe.transform(X_train)


In [ ]:
oe = OrdinalEncoder(categories=[['Poor','Average','Good'],['School','UG','PG']])

Fitting **learns** the category order for each column (validating it matches what was passed in `categories=`), without changing any data yet.

In [ ]:
oe.fit(X_train)

In [ ]:
X_train = oe.transform(X_train)

In [ ]:
X_train

`oe.categories_` confirms the exact order the encoder is using for each column — worth double-checking this matches what you intended, since a mismatched order would quietly assign wrong ranks.

In [ ]:
oe.categories_

### ⚠️ Missing Step: `X_test` Was Never Transformed

The original notebook fits the encoder and transforms `X_train`, but never applies that same transformation to `X_test` — meaning `X_test` would still be sitting as raw text (`'Poor'`, `'Good'`, etc.) if you tried to evaluate a model on it later. Exactly like the scaler notebooks: **fit once on train, transform both train and test.**

In [ ]:
X_test = oe.transform(X_test)
X_test


## 2. Encoding the Target — `LabelEncoder`

`LabelEncoder` is specifically for a **single column** (typically the target `y`), mapping each unique category to an integer (`0, 1, 2, ...`) alphabetically. Unlike `OrdinalEncoder`, it doesn't take a `categories=` argument — it isn't meant to encode a feature where a specific *order* matters, since it's usually applied to the target rather than an input.

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()

In [ ]:
le.fit(y_train)

`.fit()` learns the target's unique classes from `y_train` — `le.classes_` (below) confirms which text label got mapped to which integer.

In [ ]:
le.classes_

In [ ]:
y_train = le.transform(y_train)
y_test = le.transform(y_test)

Note this cell correctly transforms **both** `y_train` and `y_test` with the *same* fitted encoder — the right pattern (learn from train, apply consistently to both), unlike the `X_test` gap fixed above.

In [ ]:
y_train

## 3. When Neither Encoder Fits — Nominal (Unordered) Categories

Both encoders above assume some notion of order or a single target column. For a **feature** with categories that have **no natural order** (e.g. `gender`, or a `city` column) — where `OrdinalEncoder`-style integer ranks would wrongly imply one category is "more" than another — the standard tool instead is **one-hot encoding**: a separate binary (0/1) column per category.

In [ ]:
# pandas' quick version, handy for exploration:
pd.get_dummies(df[['review']], drop_first=True)

# scikit-learn's version, preferred inside an ML pipeline (fit on train, transform both train and test):
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe.fit(X_train[['review']])


## Summary

| Situation | Encoder | Key argument |
|---|---|---|
| Feature with a natural order | `OrdinalEncoder` | `categories=[[...]]` — must specify the order per column |
| Target column (`y`) | `LabelEncoder` | none needed — always alphabetical |
| Feature with **no** natural order | `OneHotEncoder` / `pd.get_dummies()` | one binary column per category |

### The pattern every encoder (and scaler) follows
1. `train_test_split()` **first** — before any fitting
2. `.fit()` **only** on `X_train` / `y_train` — never let the test set influence what's learned
3. `.transform()` **both** `X_train`/`y_train` *and* `X_test`/`y_test` — a fitted-but-not-applied test set is a gap worth double-checking for, as it was here